# Marathon Runner Profiles: From Preparation to Performance

## A visual story of 80,000 runners

**Main question:** Can distinct runner profiles be identified from training behaviour, physical condition, recovery and psychological preparation—and how do those profiles differ in marathon performance?

This notebook translates the validated clustering analysis into a concise visual story. The profiles are descriptive, not causal, and the dataset is synthetic.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')
COLORS = ['#5B8FF9', '#61DDAA', '#F6BD16', '#E8684A']

def find_data_dir():
    for candidate in [Path('../data'), Path('data')]:
        if (candidate / 'clean.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find data/clean.csv')

DATA = find_data_dir()
clean = pd.read_csv(DATA / 'clean.csv')
profiles = pd.read_csv(DATA / 'profiles.csv')
summary = pd.read_csv(DATA / 'profile_summary.csv')
df = clean.merge(profiles, on='runner_id', how='inner')
name_map = summary.set_index('profile')['profile_name'].to_dict()
df['profile_name'] = df['profile'].map(name_map)
order = summary.sort_values('profile')['profile_name'].tolist()

assert len(df) == len(clean) == 80_000
assert df['profile'].notna().all() and df['profile'].nunique() == 4
print(f'{len(df):,} runners | {df.shape[1]} variables | 4 profiles')

## 1. The expectation

Marathon performance should reflect several dimensions: training volume and consistency, aerobic capacity and experience, recovery, and psychological preparation. We therefore searched for profiles using **pre-race variables only**—never the final result.

The outcome measures below are used only after clustering to compare performance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sample = df['actual_finish_time_minutes'].dropna()
sns.histplot(sample, bins=45, color='#5B8FF9', ax=ax)
ax.axvline(sample.mean(), color='#E8684A', lw=3, label=f'Mean: {sample.mean():.1f} min')
ax.set(title='Marathon performance varies widely', xlabel='Actual finish time (minutes)', ylabel='Runners')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Do four natural groups really exist?

The validated solution uses K-means with $k=4$. It is highly reproducible across split samples (**ARI ≈ 0.934**), but its silhouette score is low (**≈ 0.12**). This distinction matters:

- The segmentation is stable enough to describe the data.
- The runners do not form four sharply separated natural species.
- The profiles should be interpreted as **regions along a continuum**.

In [ ]:
diagnostics = pd.DataFrame({
    'Measure': ['Split-half stability (ARI)', 'Cluster separation (silhouette)', 'Finish-time variance explained'],
    'Value': [0.934, 0.120, 0.272]
})
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=diagnostics, x='Value', y='Measure', palette=['#61DDAA','#E8684A','#5B8FF9'], ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel('Score / proportion')
ax.set_ylabel('')
ax.set_title('Stable assignment, but weak separation')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=5)
plt.tight_layout()
plt.show()

## 3. The four runner profiles

The cluster names summarize the variables that most clearly distinguish each segment. Cluster size is itself informative: almost half of the sample belongs to the low-frequency beginner region.

In [ ]:
plot_summary = summary.sort_values('profile').copy()
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_summary, x='share', y='profile_name', palette=COLORS, ax=ax)
ax.set(title='Most runners fall into the low-frequency beginner profile', xlabel='Share of runners (%)', ylabel='')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=5)
ax.set_xlim(0, plot_summary['share'].max() + 8)
plt.tight_layout()
plt.show()

In [ ]:
features = {
    'Weekly mileage': 'weekly_mileage_miles',
    'Runs per week': 'runs_per_week',
    'Running experience': 'running_experience_months',
    'VO2 max': 'vo2_max',
    'Training adherence': 'training_adherence_pct',
    'Recovery': 'recovery_score'
}
profile_means = df.groupby('profile_name')[[*features.values()]].mean().reindex(order)
z = (profile_means - df[[*features.values()]].mean()) / df[[*features.values()]].std()
z.columns = features.keys()
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(z, cmap='vlag', center=0, annot=True, fmt='.1f', linewidths=.5, ax=ax)
ax.set(title='What distinguishes the profiles?\n(Standard deviations from the overall mean)', xlabel='', ylabel='')
plt.tight_layout()
plt.show()

## 4. Performance: speed tells only part of the story

The **Experienced high-load** runners are fastest. However, absolute speed is not the same as meeting an individual target. To assess expectation management, we also compare the gap between actual and target time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=plot_summary, y='profile_name', x='finish_time_mean', palette=COLORS, ax=axes[0])
axes[0].set(title='Experienced high-load runners are fastest', xlabel='Mean finish time (minutes)', ylabel='')
axes[0].set_xlim(220, 315)
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.1f', padding=4)

sns.barplot(data=plot_summary, y='profile_name', x='goal_gap_mean', palette=COLORS, ax=axes[1])
axes[1].set(title='High aerobic capacity runners miss targets by less', xlabel='Mean target gap (minutes)', ylabel='')
axes[1].set_xlim(0, 46)
for c in axes[1].containers: axes[1].bar_label(c, fmt='+%.1f', padding=4)
plt.tight_layout()
plt.show()

> **Central finding:** the fastest profile is not the profile that best meets expectations. Experienced high-load runners finish about **33.6 minutes faster** than low-frequency beginners, while high-aerobic-capacity runners achieve the smallest mean target gap.

## 5. Outcomes confirm the difference

Medal rate and DNF rate point in the same general direction, but they add nuance. High aerobic capacity runners have the highest medal rate, while both physically strong profiles have the lowest dropout rates.

In [ ]:
long = plot_summary.melt(id_vars='profile_name', value_vars=['medal_rate','dnf_rate'], var_name='Outcome', value_name='Rate')
long['Outcome'] = long['Outcome'].map({'medal_rate':'Medal rate','dnf_rate':'DNF rate'})
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, outcome in zip(axes, ['Medal rate','DNF rate']):
    part = long[long['Outcome'] == outcome]
    sns.barplot(data=part, y='profile_name', x='Rate', order=order, palette=COLORS, ax=ax)
    ax.set(title=outcome, xlabel='Percent of runners', ylabel='')
    for c in ax.containers: ax.bar_label(c, fmt='%.1f%%', padding=4)
plt.tight_layout()
plt.show()

## 6. The psychological puzzle

Motivation, mental preparation, sleep and recovery were conceptually important to our question. Yet in this synthetic dataset, their direct relationships with finish time are weak. Physical capacity and accumulated experience dominate the observable performance pattern.

This is an association, not a causal result. Weak effects may reflect how the synthetic data were generated rather than how real runners behave.

In [ ]:
candidate = {
    'VO2 max':'vo2_max', 'Personal best':'personal_best_minutes',
    'Experience':'running_experience_months', 'Weekly mileage':'weekly_mileage_miles',
    'Training adherence':'training_adherence_pct', 'Recovery':'recovery_score',
    'Sleep':'sleep_hours_avg', 'Motivation':'motivation_level',
    'Mental preparation':'mental_preparation_score'
}
corr = pd.Series({label: df[col].corr(df['actual_finish_time_minutes']) for label, col in candidate.items()})
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#E8684A' if v > 0 else '#5B8FF9' for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', lw=1)
ax.invert_yaxis()
ax.set(title='Physical performance signals dominate simple correlations', xlabel='Correlation with finish time', ylabel='')
for i, v in enumerate(corr): ax.text(v + (0.01 if v >= 0 else -0.01), i, f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right')
plt.tight_layout()
plt.show()

## 7. What we can—and cannot—conclude

### What the analysis supports

1. Four stable, interpretable preparation profiles can be constructed.
2. Experienced high-load runners are the fastest on average.
3. High aerobic capacity runners show the smallest target gap and highest medal rate.
4. Stronger preparation profiles have lower DNF rates.

### What it does not support

- Four sharply separated, naturally occurring runner types.
- Causal claims that a profile or single behaviour produces an outcome.
- Strong real-world conclusions about psychology, because the dataset is synthetic.

The clusters explain about **27.2%** of finish-time variation: useful, but far from the whole race story.

In [ ]:
display_summary = plot_summary[['profile_name','n','share','finish_time_mean','goal_gap_mean','medal_rate','dnf_rate']].copy()
display_summary.columns = ['Profile','Runners','Share (%)','Finish time (min)','Target gap (min)','Medal rate (%)','DNF rate (%)']
display_summary

## Final takeaway

> Marathon runners differ along a continuum of preparation. More experience and training load are associated with faster finishes, but aerobic capacity aligns more closely with meeting personal expectations. Profiles are therefore useful storytelling tools—not proof of four natural kinds of runners.

### Recommended hand-off for the team

- Select 5–7 visuals from this notebook for the presentation.
- Convert the narrative into slides and speaker notes.
- Add a short methods slide referencing `03_profiles.ipynb`.
- Keep the synthetic-data limitation visible in the conclusion.